<a href="https://colab.research.google.com/github/Deeptanshu-Samanta/Smart-Plant-Watering-IoT-System/blob/main/Smart_PlantWaterer_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
df=pd.read_csv("/content/Dataset_watering.csv")
col=["Air temperature (C)","Wind speed (Km/h)","Air humidity (%)","Wind gust (Km/h)","Pressure (KPa)"]
for i in col:
  df[i].fillna(df[i].median(),inplace=True)

df["Status"] = df["Status"].str.strip().str.lower()
df["Status"] = df["Status"].map({"on": 1, "off": 0})

col=["Soil Moisture","Temperature"," Soil Humidity","Time","Air temperature (C)","Wind speed (Km/h)","Air humidity (%)","Wind gust (Km/h)","Pressure (KPa)"]
y=df["Status"]
x=df[col]
x_train=x
y_train=y

model1=LogisticRegression(max_iter=3000)
model1.fit(x_train,y_train)

model2=RandomForestClassifier(n_estimators=450,max_depth=13,random_state=30,class_weight="balanced")
model2.fit(x_train,y_train)

import joblib
joblib.dump(model1,"logistic_plantWater.pkl")
joblib.dump(model2,"randomForest_plantWater.pkl")

/tmp/ipython-input-1439/1209487866.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[i].fillna(df[i].median(),inplace=True)


['randomForest_plantWater.pkl']

In [ ]:
def decide_watering(soil_moisture,temperature,soil_humidity,time,air_temp,wind_speed,air_humidity,wind_gust,pressure):

    input_data = [[soil_moisture,temperature,soil_humidity,time,air_temp,wind_speed,air_humidity,wind_gust,pressure]]

    rf_pred = model2.predict(input_data)[0]
    lr_pred = model1.predict(input_data)[0]

    if soil_moisture < 20 and air_humidity < 30:
        return 1, rf_pred, lr_pred

    if soil_moisture > 65:
        return 0, rf_pred, lr_pred

    if time < 5 or time > 20:
        return 0, rf_pred, lr_pred

    final_decision = int(rf_pred == 1 or lr_pred == 1)
    return final_decision, rf_pred, lr_pred

In [ ]:
import requests

API_KEY = "YOUR API KEY"
CITY = "YOUR CITY"   # change to your city

url = f"http://api.weatherapi.com/v1/current.json?key={API_KEY}&q={CITY}"

response = requests.get(url)
data = response.json()

print("Response by JSON:",data)

weather_data = {
    "air_temp": data["current"]["temp_c"],
    "air_humidity": data["current"]["humidity"],
    "pressure": data["current"]["pressure_mb"] / 10,  # mb → kPa
    "wind_speed": data["current"]["wind_kph"],
    "wind_gust": data["current"]["gust_kph"]
}

print(weather_data)


Response by JSON: {'location': {'name': 'New York', 'region': 'New York', 'country': 'United States of America', 'lat': 40.7142, 'lon': -74.0064, 'tz_id': 'America/New_York', 'localtime_epoch': 1770364034, 'localtime': '2026-02-06 02:47'}, 'current': {'last_updated_epoch': 1770363900, 'last_updated': '2026-02-06 02:45', 'temp_c': -10.6, 'temp_f': 12.9, 'is_day': 0, 'condition': {'text': 'Clear', 'icon': '//cdn.weatherapi.com/weather/64x64/night/113.png', 'code': 1000}, 'wind_mph': 6.3, 'wind_kph': 10.1, 'wind_degree': 359, 'wind_dir': 'N', 'pressure_mb': 1013.0, 'pressure_in': 29.9, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 77, 'cloud': 0, 'feelslike_c': -16.0, 'feelslike_f': 3.2, 'windchill_c': -15.4, 'windchill_f': 4.3, 'heatindex_c': -11.2, 'heatindex_f': 11.8, 'dewpoint_c': -18.3, 'dewpoint_f': -0.9, 'vis_km': 16.0, 'vis_miles': 9.0, 'uv': 0.0, 'gust_mph': 13.2, 'gust_kph': 21.2, 'short_rad': 0, 'diff_rad': 0, 'dni': 0, 'gti': 0}}
{'air_temp': -10.6, 'air_humidity': 77, 'pres

In [ ]:
from datetime import datetime
decision, rf, lr = decide_watering(
    soil_moisture=73,
    temperature=32.5,
    soil_humidity=45,
    time=datetime.now().hour,
    air_temp=weather_data["air_temp"],
    wind_speed=weather_data["wind_speed"],
    air_humidity=weather_data["air_humidity"],
    wind_gust=weather_data["wind_gust"],
    pressure=weather_data["pressure"]
)
print("RF:", rf)
print("LR:", lr)
print("FINAL:", "WATER" if decision else "DO NOT WATER")


RF: 0
LR: 1
FINAL: DO NOT WATER


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
